# A pixel's vertical profile

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huizhml/global-vegetation-structure/blob/master/notebooks/02_vertical_profile.ipynb)

A canopy height map gives one number per pixel. This dataset gives 101: RH0 … RH100, the heights by which 0 … 100 % of the returned lidar energy has accumulated. Together they are a cumulative vertical profile.

Here we read all 101 at two places in the Spessart, Germany, ~600 m apart on the same transect the explorer's guided tour shows. Both are about 23 m tall (RH98), but they are built very differently lower down.

In [ ]:
%pip install -q rasterio mgrs

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.warp import transform
from rasterio.windows import from_bounds

# Reading over HTTP: skip directory listings, merge neighbouring byte ranges.
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
os.environ["GDAL_HTTP_MERGE_CONSECUTIVE_RANGES"] = "YES"

STORE = "https://data.source.coop/geoai-ucph/gvsm"
YEAR = 2020
NODATA = 32767


def tile_url(tile, rh, year=YEAR):
    """10 m file: one per Sentinel-2 (MGRS) tile and RH. Q1 = the median prediction."""
    return f"{STORE}/{year}/{tile}/RH{rh}_Q1.tif"


def mosaic_url(rh, year=YEAR):
    """~1 km global overview: one file per RH."""
    return f"{STORE}/mosaics/{year}/RH{rh}_Q1.tif"


def read_box(url, lon, lat, half_m):
    """Read a square of 2*half_m metres around (lon, lat) from a 10 m tile, in metres.

    Only the bytes of that window are fetched: the files are Cloud-Optimized GeoTIFFs.
    Values are stored as decimetres with 32767 as nodata.
    """
    with rasterio.open(url) as src:
        xs, ys = transform("EPSG:4326", src.crs, [lon], [lat])
        x, y = xs[0], ys[0]
        win = from_bounds(x - half_m, y - half_m, x + half_m, y + half_m, src.transform)
        data = src.read(1, window=win.round_offsets().round_lengths()).astype("float32")
    data[data == NODATA] = np.nan
    return data / 10

In [ ]:
# Two stands on the tour's transect (49.937° N), each read as the mean of a ~100 m box.
STANDS = {
    "A": (9.3080, 49.937),
    "B": (9.3164, 49.937),
}
TILE = "32UNA"
RHS = list(range(0, 101))


def mean_rh(rh, lon, lat):
    return np.nanmean(read_box(tile_url(TILE, rh), lon, lat, half_m=50))


profiles = {}
for name, (lon, lat) in STANDS.items():
    # 101 small reads over HTTP; threads keep it to tens of seconds.
    with ThreadPoolExecutor(max_workers=16) as pool:
        profiles[name] = np.array(list(pool.map(lambda rh: mean_rh(rh, lon, lat), RHS)))
    p = profiles[name]
    print(f"{name}: RH98 = {p[98]:.1f} m, RH50 = {p[50]:.1f} m, RH25 = {p[25]:.1f} m")

## The RH curve

Height against *n*: how high you have to go before *n* % of the energy has come back. A curve that rises early means much of the energy returns from near the ground; one that stays flat and then climbs means the energy comes back from up in the crowns.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, p in profiles.items():
    ax.plot(RHS, p, label=f"stand {name}", lw=2)
ax.axvline(25, color="grey", ls=":", lw=1)
ax.set_xlabel("RH n (% of returned energy)")
ax.set_ylabel("height (m)")
ax.legend()
ax.set_title("Same canopy height, different structure")
plt.show()

## Where the energy is

The step from RH*n* to RH*n+1* holds 1 % of the energy, so the spacing of the RH heights shows where the energy is concentrated: many RH metrics packed into a few metres mean a dense layer there.

In [ ]:
bins = np.arange(0, 51)  # 1 m bins up to 50 m
fig, ax = plt.subplots(figsize=(6, 5))
for name, p in profiles.items():
    # 1 % of the energy in each step between consecutive RH heights
    share, _ = np.histogram(p[1:], bins=bins)
    ax.plot(share, bins[:-1] + 0.5, label=f"stand {name}", lw=2)
ax.set_xlabel("% of returned energy per 1 m")
ax.set_ylabel("height (m)")
ax.legend()
plt.show()